# Training — Business Entity Resolution

Runs the two-stage pipeline from `business_entity_resolution/src` one stage per cell:
normalise → block → stage-1 prune → stage-2 match → threshold tuning → final models → test prediction.

Set `DATASET` in the config cell:
- `sample`: 2% of train/test, a few minutes; use it to check the notebook runs.
- `full`: the real run; writes `business_entity_resolution/output/matching_results.tsv` and `candidate_pairs.tsv`.

Normalised tables and blocking results are cached in `business_entity_resolution/cache/<DATASET>/`.
Delete that folder after changing normalisation or blocking code.

In [1]:
import gc, json, os, pickle, subprocess, sys, time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "business_entity_resolution" / "src").exists())
SRC = ROOT / "business_entity_resolution" / "src"
sys.path.insert(0, str(SRC))

from data import load_prepared, load_ground_truth
from features import (i1_stats, i2_chunks, cheap_chunk, string_features, stack_features,
                      feature_columns, group_rank_desc, STRING_BACKEND)
from models import GBM
from metric import macro_f05, breakdown
from search import backend_name
from pipeline import (log, owner_array, prune_per_s23, decode, to_ids, tune_threshold, oof,
                      blocking_cached, predict_test)

log(f"root={ROOT}")
log(f"search backend={backend_name()}  string backend={STRING_BACKEND}")

11:54:33 root=C:\Users\Aarnav\Desktop\Amazon ML


11:54:37 search backend=torch-cuda  string backend=rapidfuzz


## Config

In [2]:
DATASET = "sample"   # "sample" | "region" (train only, no test) | "full"

DATA = {
    "sample": ROOT / "business_entity_resolution" / "dataset_sample",
    "region": ROOT / "business_entity_resolution" / "dataset_region",
    "full":   ROOT / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset",
}[DATASET]
CACHE = ROOT / "business_entity_resolution" / "cache" / DATASET
OUT = ROOT / "business_entity_resolution" / "output" if DATASET == "full" else CACHE / "output"
RUN_TEST = (DATA / "test").exists()

args = SimpleNamespace(
    data=str(DATA), cache=str(CACHE), out=str(OUT),
    frac=0.1 if DATASET == "full" else 1.0,  # share of train S1 entities used for training
    train_countries=None,                    # e.g. "us" to train on one country only
    holdout=False,                           # leave-one-country-out report (slow)
    encoder="rp", dim=256, k_scale=1.0,
    keep_per_s23=8,   # blocking: S1 candidates kept per S2/S3 record
    touch_rank=3,     # train sampling: S23 kept if a sampled S1 is in its top-N
    M=3,              # S1 candidates kept per S2/S3 record after stage 1
    folds=4, backend="auto", n_est=600, chunk_rows=2_000_000, seed=0,
)
CACHE.mkdir(parents=True, exist_ok=True)
log(f"DATASET={DATASET}  data={DATA}  run_test={RUN_TEST}  frac={args.frac}")

11:54:37 DATASET=sample  data=C:\Users\Aarnav\Desktop\Amazon ML\business_entity_resolution\dataset_sample  run_test=True  frac=1.0


## 1. Load and normalise train

In [3]:
t = time.time()
truth = load_ground_truth(args.data)
s1, s23 = load_prepared(args.data, "train", args.cache)
present, ids1 = set(s23["entity_id"]), set(s1["entity_id"])
truth = {k: v & present for k, v in truth.items() if k in ids1}
log(f"[train] S1={len(s1):,}  S23={len(s23):,}  ({time.time() - t:.0f}s)")
s1["country_n"].value_counts()

11:54:37 [train] S1=44,286  S23=207,712  (0s)


country_n
us       26422
india    17864
Name: count, dtype: int64

## 2. Blocking (per-country GPU search) and labels

In [4]:
t = time.time()
B = blocking_cached(s1, s23, args, "train")
own = owner_array(s1, s23, truth)
y_all = (own[B["i2"]] == B["i1"]).astype(np.int8)
n_true = int((own >= 0).sum())
log(f"[train] pairs={len(y_all):,} ({len(y_all) / len(s23):.2f}/S23)  "
    f"blocking recall={y_all.sum() / max(n_true, 1):.4f} of {n_true:,} links  ({time.time() - t:.0f}s)")

11:54:37 [train] blocking: 1661695 pairs (top-8 S1 per S23)


11:54:37 [train] pairs=1,661,695 (8.00/S23)  blocking recall=0.9961 of 153,667 links  (0s)


## 3. Sample training rows and build stage-1 (cheap) features

In [5]:
t = time.time()
st1 = i1_stats(B, len(s1))
rng = np.random.default_rng(args.seed)
samp1 = rng.random(len(s1)) < args.frac
if args.train_countries:
    samp1 &= s1["country_n"].isin(set(args.train_countries.split(","))).values

# keep every candidate pair of each S2/S3 record whose top-`touch_rank` candidates include a sampled S1
top = group_rank_desc(B["i2"], B["s_nm"] + B["s_ad"] + B["s_full"]) <= args.touch_rank
touch = np.zeros(len(s23), bool)
touch[B["i2"][samp1[B["i1"]] & top]] = True
parts, ys = [], []
for a, b in i2_chunks(B, args.chunk_rows):
    r = touch[B["i2"][a:b]]
    if r.any():
        parts.append(cheap_chunk(B, a, b, s1, s23, st1, rows=r))
        ys.append(y_all[a:b][r])
X = pd.concat(parts, ignore_index=True)
y = np.concatenate(ys)
del B, y_all, parts, ys, top, touch
gc.collect()

ids = s1["entity_id"].values[samp1].tolist()
c1 = feature_columns(X)
log(f"[train] sampled S1={len(ids):,}  rows={len(X):,}  positives={int(y.sum()):,}  "
    f"stage-1 features={len(c1)}  ({time.time() - t:.0f}s)")

11:54:40 [train] sampled S1=44,286  rows=1,661,695  positives=153,071  stage-1 features=33  (2s)


## 4. Stage 1: out-of-fold scores, pruning, final stage-1 model

In [6]:
t = time.time()
X["p1"] = oof(X, y, c1, X["i1"].values, 3, args, "stage1")
for M in (1, 2, 3, 5):
    kept = prune_per_s23(X, "p1", M)
    log(f"  prune M1={M}: keeps {y[kept.index].sum() / max(y.sum(), 1):.4f} of blocked positives, "
        f"rows/S23={len(kept) / X['i2'].nunique():.2f}")
keep = prune_per_s23(X, "p1", args.M)
Xp = keep.reset_index(drop=True)
yp = y[keep.index]
m1 = GBM(args.backend, n_estimators=args.n_est).fit(X[c1].values, y)
del X, keep, kept
gc.collect()
log(f"[train] stage 1 done: kept {len(Xp):,} rows (M={args.M})  ({time.time() - t:.0f}s)")

11:54:53   [stage1] fold 0 done (lgbm, train rows=1107796)


11:55:06   [stage1] fold 1 done (lgbm, train rows=1107797)


11:55:21   [stage1] fold 2 done (lgbm, train rows=1107797)


11:55:21   prune M1=1: keeps 0.9929 of blocked positives, rows/S23=1.00


11:55:22   prune M1=2: keeps 0.9970 of blocked positives, rows/S23=2.00


11:55:22   prune M1=3: keeps 0.9984 of blocked positives, rows/S23=3.00


11:55:23   prune M1=5: keeps 0.9995 of blocked positives, rows/S23=5.00


11:55:40 [train] stage 1 done: kept 623,136 rows (M=3)  (61s)


## 5. Stage 2: string + stacked features, out-of-fold scores

In [7]:
t = time.time()
Xp = stack_features(Xp, "p1")
Xp = string_features(Xp, s1, s23)
c2 = feature_columns(Xp)
log(f"[train] stage-2 features {Xp.shape}  ({time.time() - t:.0f}s)")

t = time.time()
df = Xp[["i1", "i2"]].copy()
df["p"] = oof(Xp, yp, c2, Xp["i1"].values, args.folds, args, "stage2")
log(f"[train] stage-2 OOF done  ({time.time() - t:.0f}s)")

11:55:54 [train] stage-2 features (623136, 74)  (14s)


11:56:04   [stage2] fold 0 done (lgbm, train rows=467352)


11:56:13   [stage2] fold 1 done (lgbm, train rows=467352)


11:56:23   [stage2] fold 2 done (lgbm, train rows=467352)


11:56:33   [stage2] fold 3 done (lgbm, train rows=467352)


11:56:33 [train] stage-2 OOF done  (39s)


## 6. Threshold tuning and validation score (out-of-fold F0.5)

In [8]:
t = time.time()
dec, table = tune_threshold(df, s1, s23, truth, ids)
for sc, c in table[:5]:
    log(f"  decoder {sc:.4f} {c}")
pred = to_ids(decode(df, dec["t"], dec["margin"]), s1, s23)
bd = breakdown(pred, truth, ids)
log(f"[train] OOF F0.5={bd['all']:.4f}  singletons={bd['singletons']:.4f} ({bd['n_single']:,})  "
    f"with_matches={bd['with_matches']:.4f} ({bd['n_match']:,})")
ceil_idx = df[yp == 1].groupby("i1")["i2"].apply(set).to_dict()
log(f"[train] ceiling (perfect matcher on kept candidates): {macro_f05(to_ids(ceil_idx, s1, s23), truth, ids):.4f}")
samp = s1[s1["entity_id"].isin(set(ids))]
for c, g in samp.groupby("country_n"):
    cid = g["entity_id"].tolist()
    log(f"[train] country '{c}': OOF F0.5={macro_f05(pred, truth, cid):.4f} (n={len(cid):,})")
log(f"({time.time() - t:.0f}s)")

11:57:05   decoder 0.9908 {'t': 0.75, 'margin': 0.2}


11:57:05   decoder 0.9907 {'t': 0.85, 'margin': 0.2}


11:57:05   decoder 0.9907 {'t': 0.75, 'margin': 0.1}


11:57:05   decoder 0.9907 {'t': 0.85, 'margin': 0.1}


11:57:05   decoder 0.9907 {'t': 0.8, 'margin': 0.2}


11:57:05 [train] OOF F0.5=0.9908  singletons=0.9837 (2,393)  with_matches=0.9912 (41,893)


11:57:06 [train] ceiling (perfect matcher on kept candidates): 0.9981


11:57:06 [train] country 'india': OOF F0.5=0.9882 (n=17,864)


11:57:06 [train] country 'us': OOF F0.5=0.9925 (n=26,422)


11:57:06 (33s)


## 7. Leave-one-country-out check (only if `args.holdout`)

In [9]:
if args.holdout:
    cn_row = s1["country_n"].values[Xp["i1"].values]
    for c in sorted(set(cn_row)):
        tr = cn_row != c
        if tr.all() or (~tr).all():
            continue
        m = GBM(args.backend, n_estimators=args.n_est).fit(Xp[tr][c2].values, yp[tr])
        d = df[~tr].copy()
        d["p"] = m.predict_proba(Xp[~tr][c2].values)
        cid = samp[samp["country_n"] == c]["entity_id"].tolist()
        at_global = macro_f05(to_ids(decode(d, dec["t"], dec["margin"]), s1, s23), truth, cid)
        best = max((macro_f05(to_ids(decode(d, t), s1, s23), truth, cid), t) for t in np.arange(0.3, 0.91, 0.05))
        log(f"[train] holdout '{c}': F0.5={at_global:.4f} at global t; best t={best[1]:.2f} -> {best[0]:.4f}")
else:
    log("holdout skipped")

11:57:06 holdout skipped


## 8. Final stage-2 model and saved artefacts

In [10]:
t = time.time()
m2 = GBM(args.backend, n_estimators=args.n_est).fit(Xp[c2].values, yp)
with open(CACHE / "models.pkl", "wb") as f:
    pickle.dump(dict(m1=m1, m2=m2, c1=c1, c2=c2, cfg=dec), f)
with open(CACHE / "cv_summary.json", "w") as f:
    json.dump(dict(cfg=dec, oof=bd, M=args.M, string_backend=STRING_BACKEND, args=vars(args)),
              f, default=float, indent=1)
oof_frame = Xp.copy(); oof_frame["y"] = yp; oof_frame["p"] = df["p"].values
with open(CACHE / "oof_train.pkl", "wb") as f:
    pickle.dump(dict(X=oof_frame, s1=s1, s23=s23, truth=truth, cfg=dec, c2=c2, ids=ids), f, protocol=4)
del Xp, oof_frame, s1, s23
gc.collect()
log(f"[train] saved models.pkl, cv_summary.json, oof_train.pkl to {CACHE}  ({time.time() - t:.0f}s)")

11:57:22 [train] saved models.pkl, cv_summary.json, oof_train.pkl to C:\Users\Aarnav\Desktop\Amazon ML\business_entity_resolution\cache\sample  (16s)


## 9. Test prediction → `matching_results.tsv`, `candidate_pairs.tsv`

In [11]:
if RUN_TEST:
    # free train-side state (the ground-truth dict alone is ~2 GB on the full data)
    for name in ("truth", "own", "df", "pred", "samp", "table", "y", "yp", "st1", "samp1", "present", "ids1", "ids"):
        globals().pop(name, None)
    gc.collect()
    t = time.time()
    predict_test(args, m1, m2, c1, c2, dec)
    log(f"[test] done  ({time.time() - t:.0f}s)")
else:
    log(f"no test split in {DATA}; skipped")

11:57:22 [test] S1=34915 S23=199441


11:57:22 blocking: backend=torch-cuda encoder=rp dim=256


11:57:22   country 'france': S1=5218 S23=28728


11:57:24     view nm: k_rev=5 k_fwd=12 2s


11:57:26     view ad: k_rev=5 k_fwd=12 2s


...

11:59:07 [test] wrote C:\Users\Aarnav\Desktop\Amazon ML\business_entity_resolution\cache\sample\output: 4473 S1 with matches (5162 links), 30442 predicted singletons; candidates/S1=17.1


11:59:07 [test] country 'france': n=5218 with-match=0.243 links/S1=0.32


11:59:07 [test] country 'india': n=16385 with-match=0.113 links/S1=0.12


11:59:07 [test] country 'us': n=13312 with-match=0.102 links/S1=0.11


11:59:07 [test] done  (105s)


## 10. Validate the submission files

In [12]:
if RUN_TEST:
    files = ["--matching", str(OUT / "matching_results.tsv"), "--candidate", str(OUT / "candidate_pairs.tsv"),
             "--test-dir", str(DATA / "test")]
    official = DATA.parent / "utils" / "validate_submission.py"
    script = official if DATASET == "full" and official.exists() else SRC / "check_submission.py"
    r = subprocess.run([sys.executable, str(script), *files], capture_output=True, text=True)
    print(f"{script.name} exit={r.returncode}")
    print(r.stdout[-3000:], r.stderr[-3000:])

check_submission.py exit=0
PASS
 
